# 03 · Profile: OEWS wages (BLS metro, 2018–2025) and NYS DOL regional wages (2026)

**Method:** `docs/framework.md` §3 (the 10-section exploration template). Cards: `docs/source-cards/bls_oews_metro.md`, `docs/source-cards/nysdol_oews_regions.md`.

**Why these sources matter.** Both come from the same survey, OEWS (Occupational Employment and Wage Statistics), but they answer different questions:
- **BLS OEWS metro** gives employment and the full wage distribution (10th–90th percentile) for the New York metro area, every year since 2018. That's brief Q1 (jobs over time) and Q4 (pay over time).
- **NYS DOL regional wages** re-cuts the same survey for New York City alone (the five boroughs) and adds an entry wage and an experienced wage. That's brief Q4's "starting pay in NYC", the number a graduate cares about.

**What this notebook does:** reads the raw files, checks grain, codes, coverage and arithmetic, ties the numbers to BLS's own release, and ends with an issues list and two proposed staging tables. It changes nothing.

Every read uses `dtype=str, keep_default_na=False`. Reading the eight OEWS workbooks takes about 2–3 minutes.

In [1]:
import io
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NY_MSA = "35620"  # New York-Newark-Jersey City metro area (OMB CBSA code)
WAGE_COLS = ["H_MEAN", "A_MEAN", "H_PCT10", "H_PCT25", "H_MEDIAN", "H_PCT75", "H_PCT90",
             "A_PCT10", "A_PCT25", "A_MEDIAN", "A_PCT75", "A_PCT90"]

## 0. Source header

| | **BLS OEWS metro** (`bls_oews_metro`) | **NYS DOL regional wages** (`nysdol_oews_regions`) |
|---|---|---|
| **Publisher** | U.S. Bureau of Labor Statistics | NYS Department of Labor, Division of Research & Statistics |
| **Dataset** | OEWS May *YYYY* estimates, all metro areas (`oesmYYma.zip`, file `MSA_MYYYY_dl.xlsx`) | *Occupational Wages*, data.ny.gov `gkgz-nw24` |
| **How it was collected** | **Survey of employers** (~1.1M establishments over 3 years). Each May estimate pools six semiannual panels (e.g. May 2025 = Nov 2022 → May 2025), so a year's number is a 3-year rolling sample ([BLS technical notes](https://www.bls.gov/oes/2025/may/oes_tec.htm)) | The same OEWS survey (six panels Nov 2022 → May 2025), re-tabulated for NYS DOL's 10 labor-market regions, then aged to Q1 2026 with cost-of-living adjustments ([dataset page](https://data.ny.gov/d/gkgz-nw24)) |
| **What "employment" counts** | **Wage and salary jobs**, by employer location. Excludes the self-employed (field descriptions) | Same: "part-time and full-time workers who are paid a wage or salary" |
| **Geography** | Metro area 35620, which covers NYC plus Long Island, the lower Hudson Valley and northern New Jersey (and, before 2024, Pike County PA) | New York City region = Bronx, Kings, New York, Queens, Richmond counties, i.e. the five boroughs |
| **Extra measures** | 10th/25th/50th/75th/90th percentile wages, location quotient, standard errors | **Entry wage** = mean of the bottom third of wages; experienced wage = mean of the top two thirds |
| **Editions** | Annual, May reference. 8 files, 2018–2025 | One vintage; the portal overwrites it every year (manifest note) |
| **License** | U.S. government work, public domain | NY Open Data |

In [2]:
man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
FILES = man[man.key.isin(["bls_oews_metro", "nysdol_oews_regions"])][["key", "file", "downloaded_utc", "note"]]
FILES.reset_index(drop=True)

,key,file,downloaded_utc,note
0,bls_oews_metro,jobs/bls_oews_metro/oesm18ma.zip,2026-10-12T17:03:22+00:00,"OEWS May 2018, all metro areas (NY = 35620)"
1,bls_oews_metro,jobs/bls_oews_metro/oesm19ma.zip,2026-10-12T17:03:24+00:00,"OEWS May 2019, all metro areas (NY = 35620)"
2,bls_oews_metro,jobs/bls_oews_metro/oesm20ma.zip,2026-10-12T17:03:25+00:00,"OEWS May 2020, all metro areas (NY = 35620)"
3,bls_oews_metro,jobs/bls_oews_metro/oesm21ma.zip,2026-10-12T17:03:27+00:00,"OEWS May 2021, all metro areas (NY = 35620)"
4,bls_oews_metro,jobs/bls_oews_metro/oesm22ma.zip,2026-10-12T17:03:29+00:00,"OEWS May 2022, all metro areas (NY = 35620)"
5,bls_oews_metro,jobs/bls_oews_metro/oesm23ma.zip,2026-10-12T17:03:30+00:00,"OEWS May 2023, all metro areas (NY = 35620)"
6,bls_oews_metro,jobs/bls_oews_metro/oesm24ma.zip,2026-10-12T17:03:32+00:00,"OEWS May 2024, all metro areas (NY = 35620)"
7,bls_oews_metro,jobs/bls_oews_metro/oesm25ma.zip,2026-10-12T17:03:33+00:00,"OEWS May 2025, all metro areas (NY = 35620)"
8,nysdol_oews_regions,jobs/nysdol_oews_regions/gkgz-nw24_2026-10-12.csv,2026-10-12T17:03:40+00:00,"NYS DOL wages by region incl. NYC 5 boroughs, entry/experienced wa..."


## 1. Load as text

**OEWS:** each zip holds the metro workbook (`MSA_…`), a nonmetro workbook (`BOS_…`) and a description file. Only the metro workbook is read. All ~145k rows (every metro × job) are loaded, the layout is recorded, and only the NY metro is kept. The reference year comes from the zip name (`oesm25ma` → 2025).

In [3]:
oews_files = FILES[FILES.key == "bls_oews_metro"].file.tolist()
layouts, ny_parts = {}, []
for f in oews_files:
    year = 2000 + int(Path(f).stem[4:6])
    z = zipfile.ZipFile(RAW / f)
    members = z.namelist()
    msa = next(n for n in members if Path(n).name.startswith("MSA_"))
    d = pd.read_excel(io.BytesIO(z.read(msa)), sheet_name=0, dtype=str, keep_default_na=False)
    layouts[year] = [c.strip() for c in d.columns]
    d.columns = [c.strip().upper().replace(" ", "_") for c in d.columns]
    d = d.rename(columns={"OCC_GROUP": "O_GROUP", "AREA_NAME": "AREA_TITLE"})
    ny = d[d.AREA.str.strip() == NY_MSA].copy()
    ny.insert(0, "year", year)
    ny_parts.append(ny)
    print(f"{year}  {msa:<28} {d.shape[0]:>7,} rows x {d.shape[1]} cols | NY metro rows: {len(ny):>3} | "
          f"other members: {[Path(m).name for m in members if m != msa and not m.endswith('/')]}")

oews = pd.concat(ny_parts, ignore_index=True).fillna("")  # columns a year lacks become ""

2018  oesm18ma/MSA_M2018_dl.xlsx   143,905 rows x 25 cols | NY metro rows: 773 | other members: ['BOS_M2018_dl.xlsx', 'field_descriptions.xlsx', 'file_descriptions.xlsx']


2019  oesm19ma/MSA_M2019_dl.xlsx   142,410 rows x 30 cols | NY metro rows: 759 | other members: ['BOS_M2019_dl.xlsx', 'file_descriptions.xlsx']


2020  oesm20ma/MSA_M2020_dl.xlsx   140,051 rows x 31 cols | NY metro rows: 756 | other members: ['BOS_M2020_dl.xlsx', 'file_descriptions.xlsx']


2021  oesm21ma/MSA_M2021_dl.xlsx   147,188 rows x 32 cols | NY metro rows: 804 | other members: ['BOS_M2021_dl.xlsx', 'file_descriptions.xlsx']


2022  oesm22ma/MSA_M2022_dl.xlsx   148,204 rows x 32 cols | NY metro rows: 800 | other members: ['~$file_descriptions.xlsx', '~$MSA_M2022_dl.xlsx', 'BOS_M2022_dl.xlsx', 'file_descriptions.xlsx']


2023  oesm23ma/MSA_M2023_dl.xlsx   149,591 rows x 32 cols | NY metro rows: 803 | other members: ['BOS_M2023_dl.xlsx', 'file_descriptions.xlsx']


2024  oesm24ma/MSA_M2024_dl.xlsx   150,176 rows x 32 cols | NY metro rows: 794 | other members: ['BOS_M2024_dl.xlsx', 'file_descriptions.xlsx']


2025  oesm25ma/MSA_M2025_dl.xlsx   150,023 rows x 32 cols | NY metro rows: 792 | other members: ['BOS_M2025_dl.xlsx', 'file_descriptions.xlsx']


In [4]:
# Layout drift: which columns exist in which year (raw names, stripped)
all_cols = list(dict.fromkeys(c for cols in layouts.values() for c in cols))
pd.DataFrame({y: ["✓" if c in cols else "" for c in all_cols] for y, cols in layouts.items()}, index=all_cols)

,2018,2019,2020,2021,2022,2023,2024,2025
PRIM_STATE,✓,,✓,✓,✓,✓,✓,✓
AREA,✓,,✓,✓,✓,✓,✓,✓
AREA_NAME,✓,,,,,,,
OCC_CODE,✓,,✓,✓,✓,✓,✓,✓
OCC_TITLE,✓,,✓,✓,✓,✓,✓,✓
...,...,...,...,...,...,...,...,...
OWN_CODE,,,✓,✓,✓,✓,✓,✓
O_GROUP,,,✓,✓,✓,✓,✓,✓
LOC_QUOTIENT,,,✓,✓,✓,✓,✓,✓
PCT_TOTAL,,,✓,✓,✓,✓,✓,✓


**Load findings (OEWS)**
- **The layout changes over time.** 2018 has 25 columns, with `OCC_GROUP` instead of `O_GROUP`, `LOC QUOTIENT` with a space, and no `AREA_TYPE`/`NAICS`/`OWN_CODE`. 2019 adds the industry and ownership columns, 2020 adds `PRIM_STATE` back, and 2021 adds `PCT_RPT`. Column case also changes between years. Staging needs a per-year column map, not one fixed list.
- The 2022 zip contains two Excel lock files (`~$MSA_M2022_dl.xlsx`, `~$file_descriptions.xlsx`): BLS zipped the folder while the workbooks were open. They're harmless, but a loader that globs `*.xlsx` would pick them up. Match on `MSA_` at the start of the name.

In [5]:
w_raw = pd.read_csv(RAW / FILES[FILES.key == "nysdol_oews_regions"].file.iloc[0], dtype=str, keep_default_na=False)
print(w_raw.shape)
print([repr(c)[:45] for c in w_raw.columns])
wages = w_raw.copy()
wages.columns = [c.strip() for c in wages.columns]
padded = {c: wages[c].ne(wages[c].str.strip()).sum() for c in wages.columns}
print("Cells with leading/trailing spaces:", {k: v for k, v in padded.items() if v})
wages = wages.apply(lambda s: s.str.strip())

(7036, 10)
["'Area Type'", "' Area'", "' Area Name'", "' Standard Occupational Code'", "' Occupational Title'", "' Employment'", "'Mean Wage'", "' Median Wage'", "' Entry Wage'", "' Experienced Wage                           "]
Cells with leading/trailing spaces: {}


**Load findings (NYS DOL):** 7,036 rows × 10 columns. Eight of the ten column names have stray spaces (a leading space, and about 200 trailing spaces on `Experienced Wage`). The cell values themselves are clean. Any code that selects columns by name has to strip the names first.

## 2. What one row is (grain)

**Presumed keys:** OEWS `year × AREA × OCC_CODE` (cross-industry, all ownerships). NYS DOL `Area × Standard Occupational Code`.

In [6]:
print("OEWS NY metro: duplicate (year, OCC_CODE):", oews.duplicated(["year", "OCC_CODE"]).sum())
for c in ["AREA_TITLE", "AREA_TYPE", "NAICS", "I_GROUP", "OWN_CODE"]:
    print(f"  {c:<11}", oews.groupby("year")[c].agg(lambda s: sorted(set(s))).to_dict())

print("\nNYS DOL: duplicate (Area, SOC):", wages.duplicated(["Area", "Standard Occupational Code"]).sum())
display(wages.groupby(["Area Type", "Area", "Area Name"]).size().rename("rows").to_frame())

OEWS NY metro: duplicate (year, OCC_CODE): 0
  AREA_TITLE  {2018: ['New York-Newark-Jersey City, NY-NJ-PA'], 2019: ['New York-Newark-Jersey City, NY-NJ-PA'], 2020: ['New York-Newark-Jersey City, NY-NJ-PA'], 2021: ['New York-Newark-Jersey City, NY-NJ-PA'], 2022: ['New York-Newark-Jersey City, NY-NJ-PA'], 2023: ['New York-Newark-Jersey City, NY-NJ-PA'], 2024: ['New York-Newark-Jersey City, NY-NJ'], 2025: ['New York-Newark-Jersey City, NY-NJ']}
  AREA_TYPE   {2018: [''], 2019: ['4'], 2020: ['4'], 2021: ['4'], 2022: ['4'], 2023: ['4'], 2024: ['4'], 2025: ['4']}
  NAICS       {2018: [''], 2019: ['000000'], 2020: ['000000'], 2021: ['000000'], 2022: ['000000'], 2023: ['000000'], 2024: ['000000'], 2025: ['000000']}
  I_GROUP     {2018: [''], 2019: ['cross-industry'], 2020: ['cross-industry'], 2021: ['cross-industry'], 2022: ['cross-industry'], 2023: ['cross-industry'], 2024: ['cross-industry'], 2025: ['cross-industry']}
  OWN_CODE    {2018: [''], 2019: ['1235'], 2020: ['1235'], 2021: ['1235'],

rows
Area Type Area Area Name             
1         36   New York State     803
10        1    Capital Region     657
          10   Western New York   669
          2    Central New York   601
          3    Finger Lakes       653
          4    Hudson Valley      665
          5    Long Island        678
          6    Mohawk Valley      527
          7    New York City      702
          8    North Country      484
          9    Southern Tier      597

**Grain findings**
- **OEWS:** the key `year × OCC_CODE` is unique for the NY metro in every year . Every row is the cross-industry (`NAICS 000000`), all-ownership (`1235`) estimate, so there's nothing to filter.
- The NY metro was redefined in 2024. The title changes from *New York-Newark-Jersey City, NY-NJ-PA* to *…, NY-NJ*. The OMB 2023 delineations ([Bulletin 23-01](https://www.bls.gov/oes/2025/may/oes_tec.htm)) removed Pike County, PA, but the code stayed 35620. That's small in jobs, but it's a geography break the trend line should footnote.
- **NYS DOL:** `Area × SOC` is unique . 11 areas: New York State (`Area Type 1`) plus 10 regions (`Area Type 10`). NYC is `Area = 7`, 702 rows. One row = one job in one region, for one vintage.

## 3. Column profile

OEWS marks missing or capped values with symbols instead of leaving them blank. Count every non-numeric value per column and year:

In [7]:
num = r"-?\d+(\.\d+)?"
prof = {}
for y, g in oews.groupby("year"):
    row = {}
    for c in ["TOT_EMP", "EMP_PRSE", "JOBS_1000", "LOC_QUOTIENT"] + WAGE_COLS:
        v = g[c].str.strip()
        bad = v[~v.str.fullmatch(num)].value_counts()
        row[c] = ", ".join(f"{k or '(blank)'} ×{n}" for k, n in bad.items())
    prof[y] = row
pd.DataFrame(prof)

,2018,2019,2020,2021,2022,2023,2024,2025
TOT_EMP,** ×21,** ×20,** ×27,** ×24,** ×16,** ×27,** ×29,** ×27
EMP_PRSE,** ×21,** ×20,** ×27,** ×24,** ×16,** ×27,** ×29,** ×27
JOBS_1000,** ×21,** ×20,** ×27,** ×24,** ×16,** ×27,** ×29,** ×27
LOC_QUOTIENT,** ×21,** ×20,** ×27,** ×24,** ×16,** ×27,** ×29,** ×27
H_MEAN,* ×66,* ×63,* ×66,"* ×65, # ×1",* ×63,* ×66,* ×61,* ×64
A_MEAN,* ×15,* ×12,* ×16,"* ×11, # ×1",* ×11,* ×15,* ×8,* ×12
H_PCT10,* ×66,* ×63,* ×66,"* ×65, # ×1",* ×63,* ×66,* ×61,* ×64
H_PCT25,"* ×66, # ×2",* ×63,* ×66,"* ×65, # ×6","* ×63, # ×2",* ×66,* ×61,* ×64
H_MEDIAN,"* ×66, # ×7","* ×63, # ×7","* ×66, # ×7","* ×65, # ×19","* ×63, # ×9","* ×66, # ×11","* ×61, # ×12",* ×64
H_PCT75,"* ×66, # ×24","* ×63, # ×27","* ×66, # ×27","* ×65, # ×45","* ×63, # ×24","* ×66, # ×26","* ×61, # ×27",* ×64


**OEWS profile findings.** The publisher's notes (sheet *Field Descriptions*) define the symbols:

| Symbol | Meaning | Where |
|---|---|---|
| `**` | employment estimate not available | `TOT_EMP`, `EMP_PRSE`, `JOBS_1000`, `LOC_QUOTIENT` |
| `*` | wage estimate not available | all wage columns |
| `#` | wage ≥ the top-code ($100/hr until 2021, $115/hr = $239,200/yr from 2022) | upper percentiles, plus means and medians for the best-paid jobs |

- `` rows are jobs with a wage but no employment count**, about 20–30 per year. They need `NULL` employment, not zero.
- `H_*` columns have more `*` than `A_*` (≈65 vs ≈12). These are the annual-only jobs (teachers, pilots), flagged in the `ANNUAL` column. The reverse is `HOURLY` (actors, musicians).
- 2025 has no `#` at all. The same check in §4 shows why that's a problem.

In [8]:
nyc_w = wages[wages["Area Name"] == "New York City"]
cols = ["Employment", "Mean Wage", "Median Wage", "Entry Wage", "Experienced Wage"]
pd.DataFrame({
    "blank (NYC)": [(nyc_w[c] == "").sum() for c in cols],
    "blank (all areas)": [(wages[c] == "").sum() for c in cols],
    "non-numeric": [(~wages[c].str.fullmatch(num) & (wages[c] != "")).sum() for c in cols],
}, index=cols)

,blank (NYC),blank (all areas),non-numeric
Employment,117,1092,0
Mean Wage,0,0,0
Median Wage,0,0,0
Entry Wage,0,0,0
Experienced Wage,6,78,0


**NYS DOL profile findings.** No symbols: suppression shows up as a blank. 117 NYC jobs have wages but no employment count. 6 have no experienced wage, e.g. Chief Executives, where the top two thirds can't be published. Every non-blank value is a whole number.

## 4. Codes, hierarchy and special values

**(a) Which SOC edition?** Three marker codes tell the editions apart: `15-1132` exists only in SOC 2010, `15-1256` only in the transitional "hybrid" list, and `15-1252` only in SOC 2018.

In [9]:
markers = {"15-1132 (SOC 2010)": "15-1132", "15-1256 (hybrid)": "15-1256", "15-1252 (SOC 2018)": "15-1252"}
ed = pd.DataFrame({k: oews.groupby("year").OCC_CODE.agg(lambda s, c=v: c in set(s)) for k, v in markers.items()})
ed["codes ending in 8"] = oews[oews.OCC_CODE.str.fullmatch(r"\d\d-\d{3}8")].groupby("year").size().reindex(ed.index, fill_value=0)
ed["O_GROUP values"] = oews.groupby("year").O_GROUP.agg(lambda s: dict(s.value_counts()))
ed

,15-1132 (SOC 2010),15-1256 (hybrid),15-1252 (SOC 2018),codes ending in 8,O_GROUP values
year,,,,,
2018,True,False,False,9,"{'detailed': 750, 'major': 22, 'total': 1}"
2019,False,True,False,25,"{'detailed': 736, 'major': 22, 'total': 1}"
2020,False,True,False,25,"{'detailed': 733, 'major': 22, 'total': 1}"
2021,False,False,True,9,"{'detailed': 781, 'major': 22, 'total': 1}"
2022,False,False,True,9,"{'detailed': 777, 'major': 22, 'total': 1}"
2023,False,False,True,9,"{'detailed': 780, 'major': 22, 'total': 1}"
2024,False,False,True,9,"{'detailed': 771, 'major': 22, 'total': 1}"
2025,False,False,True,9,"{'detailed': 769, 'major': 22, 'total': 1}"


**Code findings**
- Three code systems in eight years: 2018 = SOC 2010; 2019–2020 = hybrid (BLS's transition list, with SOC-2018 structure but aggregates like `15-1256` and `13-2098` where the two editions couldn't be split); 2021–2025 = SOC 2018. A time series per job needs the SOC 2010→2018 crosswalk *and* a hybrid→2018 map (notebook 06). The hybrid aggregates are the same codes that broke the NYS DOL projections (notebook 02), so one bridge table serves both sources.
- The metro file only has two levels: `major` and `detailed`, plus the total. There are no minor or broad rows, so unlike the projections, every non-major row is a leaf and can be summed.

**(b) Top-coding.** Until 2024, wages at or above the cap were replaced by `#`. Check the highest numeric wage and the `#` count each year:

In [10]:
tc = []
for y, g in oews.groupby("year"):
    v = g[WAGE_COLS].apply(lambda s: pd.to_numeric(s, errors="coerce"))
    a = v[[c for c in WAGE_COLS if c.startswith("A_")]]
    tc.append({"year": y, "'#' cells": int((g[WAGE_COLS] == "#").sum().sum()),
               "highest annual wage shown": a.max().max(),
               "annual values ≥ $239,200": int((a >= 239_200).sum().sum()),
               "jobs with a median ≥ $239,200 or '#'": int(((v["A_MEDIAN"] >= 239_200) | (g["A_MEDIAN"] == "#")).sum())})
display(pd.DataFrame(tc).set_index("year").style.format({"highest annual wage shown": "${:,.0f}"}))

top = oews[(oews.year == 2025) & (pd.to_numeric(oews.A_MEDIAN, errors="coerce") >= 239_200)]
prev = oews[oews.year == 2024].set_index("OCC_CODE").A_MEDIAN
top.assign(A_MEDIAN_2024=top.OCC_CODE.map(prev))[["OCC_CODE", "OCC_TITLE", "A_MEDIAN_2024", "A_MEDIAN", "A_PCT90"]].head(10)

,'#' cells,highest annual wage shown,"annual values ≥ $239,200","jobs with a median ≥ $239,200 or '#'"
year,,,,
2018,167,"$263,870",5,7
2019,181,"$252,880",2,8
2020,178,"$260,390",2,8
2021,290,"$358,310",12,21
2022,189,"$405,970",13,9
2023,185,"$380,890",16,12
2024,203,"$415,630",18,13
2025,0,"$805,330",144,19


,OCC_CODE,OCC_TITLE,A_MEDIAN_2024,A_MEDIAN,A_PCT90
5491,11-1011,Chief Executives,#,301440,709670
5792,29-1022,Oral and Maxillofacial Surgeons,*,342730,392410
5793,29-1023,Orthodontists,#,281850,479770
5809,29-1151,Nurse Anesthetists,#,321030,339500
5813,29-1211,Anesthesiologists,#,460800,561000
5814,29-1212,Cardiologists,#,492150,697580
5815,29-1213,Dermatologists,#,374000,488860
5816,29-1214,Emergency Medicine Physicians,#,289440,386990
5817,29-1215,Family Medicine Physicians,#,249150,403430
5820,29-1218,Obstetricians and Gynecologists,#,329750,444380


This is an unannounced break in the series. Through 2024, every wage at or above the cap is hidden as `#` (e.g. Cardiologists' median: `#`). In 2025 the same jobs show real numbers (Cardiologists' median $492,150, Chief Executives' 90th percentile $709,670), and `#` doesn't appear anywhere. Yet the 2025 *Field Descriptions* sheet still says `#` means "≥ $115.00 per hour or $239,200 per year", and the [May 2025 technical notes](https://www.bls.gov/oes/2025/may/oes_tec.htm) don't mention any change. The 2025 notes also redefine PRSE as model error (2024: sampling error), so the estimation method changed too.

What it means for us: upper percentiles and means for top-paid jobs (physicians, executives, top finance) aren't comparable across 2024 → 2025. Treatment: in staging, a `#` becomes `NULL` + `is_top_coded = true` with the cap value kept. Growth charts for wages above the cap begin in 2025, or carry a footnote. Few CUNY majors lead straight into these jobs, so the practical impact is small, but it's a trap on the "pay over time" page.

(c) NYS DOL special values: the minimum-wage floor. The dataset page says wages below the local minimum are raised to it: $17.00/hr in NYC and Long Island, $16.00 elsewhere. At 2,080 hours that's $35,360 and $33,280.

In [11]:
fl = []
for a, g in wages.groupby("Area Name"):
    e, m = pd.to_numeric(g["Entry Wage"]), pd.to_numeric(g["Median Wage"])
    fl.append({"area": a, "lowest entry wage": e.min(), "jobs at the lowest entry wage": int((e == e.min()).sum()),
               "jobs with median at that floor": int((m == e.min()).sum())})
pd.DataFrame(fl).set_index("area")

,lowest entry wage,jobs at the lowest entry wage,jobs with median at that floor
area,,,
Capital Region,33280,11,3
Central New York,33280,17,1
Finger Lakes,33280,19,1
Hudson Valley,33280,3,0
Long Island,35360,27,2
Mohawk Valley,33280,25,2
New York City,35360,22,5
New York State,33280,2,0
North Country,33280,17,0


**Floor findings.** NYC's lowest entry wage is exactly $35,360, and 22 NYC jobs sit on it (fast food, cashiers, dishwashers…). Those aren't measured wages: they're the legal minimum written in. On a "starting pay" chart they'll all tie at the bottom. Treatment: flag `entry_wage_at_floor` so the dashboard can say "at the NYC minimum wage" instead of showing a fake precise figure.

NYC and Long Island floor at $35,360; every other region at $33,280. Only 5 NYC jobs have a *median* on the floor, so the problem is mostly in entry wages.

## 5. Coverage

**OEWS:** one area, eight years. The NY metro is twice the size of NYC, so check the totals over time:

In [12]:
tot = oews[oews.OCC_CODE == "00-0000"].set_index("year")
cov = pd.DataFrame({
    "area title": tot.AREA_TITLE,
    "jobs": pd.to_numeric(tot.TOT_EMP),
    "mean hourly": pd.to_numeric(tot.H_MEAN),
    "mean annual": pd.to_numeric(tot.A_MEAN),
    "median annual": pd.to_numeric(tot.A_MEDIAN),
    "detailed jobs published": oews[oews.O_GROUP == "detailed"].groupby("year").size(),
})
cov.style.format({"jobs": "{:,.0f}", "mean hourly": "${:.2f}", "mean annual": "${:,.0f}", "median annual": "${:,.0f}"})

,area title,jobs,mean hourly,mean annual,median annual,detailed jobs published
year,,,,,,
2018,"New York-Newark-Jersey City, NY-NJ-PA","9,498,420",$31.04,"$64,550","$47,080",750
2019,"New York-Newark-Jersey City, NY-NJ-PA","9,655,330",$32.11,"$66,790","$48,840",736
2020,"New York-Newark-Jersey City, NY-NJ-PA","8,834,370",$34.16,"$71,050","$52,020",733
2021,"New York-Newark-Jersey City, NY-NJ-PA","8,678,110",$35.65,"$74,150","$55,810",781
2022,"New York-Newark-Jersey City, NY-NJ-PA","9,185,300",$37.77,"$78,560","$57,390",777
2023,"New York-Newark-Jersey City, NY-NJ-PA","9,495,240",$39.45,"$82,050","$59,390",780
2024,"New York-Newark-Jersey City, NY-NJ","9,363,020",$40.65,"$84,560","$60,460",771
2025,"New York-Newark-Jersey City, NY-NJ","9,492,000",$41.50,"$86,310","$61,430",769


In [13]:
w_nyc_tot = nyc_w[nyc_w["Standard Occupational Code"] == "00-0000"].iloc[0]
print(f"NYS DOL, NYC (5 boroughs), wages aged to Q1 2026: {int(w_nyc_tot.Employment):>10,} jobs | mean ${int(w_nyc_tot['Mean Wage']):,}")
print(f"BLS OEWS, NY metro, May 2025:                     {int(tot.loc[2025, 'TOT_EMP']):>10,} jobs | mean ${int(tot.loc[2025, 'A_MEAN']):,}")
print(f"NYC share of metro jobs: {int(w_nyc_tot.Employment) / int(tot.loc[2025, 'TOT_EMP']):.1%}")

NYS DOL, NYC (5 boroughs), wages aged to Q1 2026:  4,603,380 jobs | mean $95,518
BLS OEWS, NY metro, May 2025:                      9,492,000 jobs | mean $86,310
NYC share of metro jobs: 48.5%


**Coverage findings**
- **The metro isn't NYC.** NYC (4.60M jobs) is 48% of the metro (9.49M). Every OEWS metro number mixes in Newark, Long Island and Westchester. The dashboard has to say "New York metro area" for these, not "NYC".
- COVID shows in employment, not in wages. Jobs fall from 9.66M (2019) to 8.68M (2021) and recover by 2023. The mean wage keeps rising through 2020–21 because the jobs lost were mostly low-paid, a composition effect, not raises. A 2019→2021 "pay growth" figure would overstate wage growth.
- Because each May estimate pools three years of panels, 2020 and 2021 still include pre-COVID responses. OEWS is a smoothed series. It's fine for trends, but don't read a single year-over-year change as a real shock.
- **NYS DOL** is a single vintage, so it has no time dimension. The portal overwrites it yearly, which is why the manifest date matters: this copy is the May 2025 survey aged to Q1 2026.

## 6. Relationships: will it join?

Three joins matter: (1) NYS DOL NYC ↔ OEWS metro on SOC (same survey, different area); (2) OEWS across years on SOC; (3) both ↔ the projections (done in notebook 02 §6: 94% of 2022-32 NYC leaves found an NYC wage row).

In [14]:
o25 = oews[oews.year == 2025].set_index("OCC_CODE")
n = nyc_w.set_index("Standard Occupational Code")
both = n.index.intersection(o25.index)
print(f"NYS DOL NYC codes: {len(n)} | found in OEWS 2025 metro: {len(both)} ({len(both)/len(n):.1%}) | not found: {sorted(set(n.index) - set(o25.index))}")

j = pd.DataFrame({
    "emp_nyc": pd.to_numeric(n.loc[both, "Employment"], errors="coerce"),
    "emp_metro": pd.to_numeric(o25.loc[both, "TOT_EMP"], errors="coerce"),
    "median_nyc_2026": pd.to_numeric(n.loc[both, "Median Wage"], errors="coerce"),
    "median_metro_2025": pd.to_numeric(o25.loc[both, "A_MEDIAN"], errors="coerce"),
})
j["nyc_share_of_metro_jobs"] = j.emp_nyc / j.emp_metro
j["nyc_median_vs_metro"] = j.median_nyc_2026 / j.median_metro_2025
display(j[["nyc_share_of_metro_jobs", "nyc_median_vs_metro"]].describe(percentiles=[.1, .5, .9]).T.round(3))
print("Jobs where NYC claims more employment than the whole metro:", int((j.nyc_share_of_metro_jobs > 1).sum()))

NYS DOL NYC codes: 702 | found in OEWS 2025 metro: 699 (99.6%) | not found: ['27-1012', '29-1029', '43-9199']


,count,mean,std,min,10%,50%,90%,max
nyc_share_of_metro_jobs,573.0,0.471,0.170,0.035,0.267,0.460,0.697,0.981
nyc_median_vs_metro,694.0,1.057,0.112,0.462,0.981,1.038,1.171,2.043


Jobs where NYC claims more employment than the whole metro: 0


In [15]:
# Code continuity across years: how many codes carry from one year to the next?
codes = oews[oews.O_GROUP == "detailed"].groupby("year").OCC_CODE.agg(set)
cont = pd.DataFrame({"detailed codes": codes.map(len),
                     "kept from previous year": [None] + [len(codes[y] & codes[y - 1]) for y in codes.index[1:]],
                     "new": [None] + [len(codes[y] - codes[y - 1]) for y in codes.index[1:]],
                     "dropped": [None] + [len(codes[y - 1] - codes[y]) for y in codes.index[1:]]})
cont

,detailed codes,kept from previous year,new,dropped
year,,,,
2018,750,NaN,NaN,NaN
2019,736,654.0,82.0,96.0
2020,733,726.0,7.0,10.0
2021,781,697.0,84.0,36.0
2022,777,769.0,8.0,12.0
2023,780,771.0,9.0,6.0
2024,771,766.0,5.0,14.0
2025,769,759.0,10.0,12.0


**Join findings**
- NYS DOL NYC → OEWS metro: all but a handful of codes match on SOC as-is. Both use SOC 2018 and the same OEWS-specific aggregates.
- NYC is a typical half of the metro for most jobs (median share ≈ 46%), but the range is wide: finance and media concentrate in NYC; logistics and warehousing sit in New Jersey. That's why metro employment can't stand in for NYC employment job by job.
- NYC medians run ≈ 4% above the metro, a mix of two things the file can't separate: NYC pays more, *and* NYS DOL aged its wages ~9 months to Q1 2026. Don't put the two sources side by side as if they were the same period.
- **Across years**, the big breaks are 2019 (SOC 2010 → hybrid) and 2021 (hybrid → SOC 2018). In SOC 2018 years, codes come and go mostly through suppression (a job falls below the publication threshold), not through code changes.

## 7. Tie-out

(a) OEWS against BLS's own release. The BLS news release *[Occupational Employment and Wages in New York-Newark-Jersey City, May 2025](https://www.bls.gov/regions/northeast/news-release/occupationalemploymentandwages_newyork.htm)* gives mean hourly wages: all occupations $41.50, computer and mathematical $67.04, healthcare practitioners and technical $62.24.

In [16]:
published = {"00-0000": 41.50, "15-0000": 67.04, "29-0000": 62.24}
check = pd.DataFrame({"published (BLS release)": published,
                      "from the file": {k: float(o25.loc[k, "H_MEAN"]) for k in published}})
check["match"] = check.iloc[:, 0] == check.iloc[:, 1]
display(check)
assert check.match.all()

,published (BLS release),from the file,match
00-0000,41.50,41.50,True
15-0000,67.04,67.04,True
29-0000,62.24,62.24,True


**(b) Internal arithmetic.** Annual wage = hourly × 2,080 where both are published. Major groups should equal the sum of their detailed jobs, except for suppressed (`**`) detail.

In [17]:
v = oews.assign(h=pd.to_numeric(oews.H_MEAN, errors="coerce"), a=pd.to_numeric(oews.A_MEAN, errors="coerce")).dropna(subset=["h", "a"])
v["diff"] = (v.a - (v.h * 2080).round(-1)).abs()
print(f"Annual mean vs hourly x 2,080: {len(v):,} rows checked, max difference ${v['diff'].max():,.0f}")

rows = []
for y, g in oews.groupby("year"):
    e = pd.to_numeric(g.TOT_EMP, errors="coerce")
    det = g.O_GROUP == "detailed"
    total = e[g.OCC_CODE == "00-0000"].iloc[0]
    rows.append({"year": y, "total jobs": total, "sum of detailed": e[det].sum(),
                 "coverage": e[det].sum() / total, "detailed with ** employment": int((g.TOT_EMP[det] == "**").sum())})
pd.DataFrame(rows).set_index("year").style.format({"total jobs": "{:,.0f}", "sum of detailed": "{:,.0f}", "coverage": "{:.1%}"})

Annual mean vs hourly x 2,080: 5,734 rows checked, max difference $20


,total jobs,sum of detailed,coverage,detailed with ** employment
year,,,,
2018,"9,498,420","9,467,660",99.7%,21
2019,"9,655,330","9,629,940",99.7%,20
2020,"8,834,370","8,807,130",99.7%,27
2021,"8,678,110","8,654,220",99.7%,24
2022,"9,185,300","9,168,710",99.8%,16
2023,"9,495,240","9,391,880",98.9%,27
2024,"9,363,020","9,286,540",99.2%,29
2025,"9,492,000","9,437,990",99.4%,27


(c) Trace one record: Software Developers (`15-1252`). Notebook 02 found the projections claim 600 NYC Software Developers. The two wage sources say:

In [18]:
sd = oews[oews.OCC_CODE.isin(["15-1252", "15-1256", "15-1132"])][["year", "OCC_CODE", "OCC_TITLE", "TOT_EMP", "A_MEDIAN", "A_PCT10", "A_PCT90"]]
display(sd)
display(nyc_w[nyc_w["Standard Occupational Code"].isin(["15-1252", "15-1253"])][
    ["Standard Occupational Code", "Occupational Title", "Employment", "Entry Wage", "Median Wage", "Experienced Wage"]])

,year,OCC_CODE,OCC_TITLE,TOT_EMP,A_MEDIAN,A_PCT10,A_PCT90
70,2018,15-1132,"Software Developers, Applications",78380,111960,69070,171100
847,2019,15-1256,Software Developers and Software Quality Assurance Analysts and Te...,100470,117680,71350,181210
1606,2020,15-1256,Software Developers and Software Quality Assurance Analysts and Te...,104470,119120,72770,182790
2370,2021,15-1252,Software Developers,93100,129050,78730,198560
3174,2022,15-1252,Software Developers,108650,136630,82100,208230
3974,2023,15-1252,Software Developers,119010,149090,86310,215420
4777,2024,15-1252,Software Developers,119610,161970,97420,220780
5570,2025,15-1252,Software Developers,121000,166830,101690,224590


,Standard Occupational Code,Occupational Title,Employment,Entry Wage,Median Wage,Experienced Wage
4663,15-1252,Software Developers,74710,121283,174663,210540
4664,15-1253,Software Quality Assurance Analysts and Testers,6100,91352,136817,164588


**Tie-out findings**
- **(a) Exact:** all three published means match the file to the cent.
- **(b)** Annual = hourly × 2,080 holds to within $20 (BLS rounds annual wages to $10). Detailed jobs cover 98.9–99.8% of the metro total every year; the rest is `**` jobs and confidential detail. Unlike the 2022-32 projections, nothing is missing.
- **(c)** Software Developers are 121,000 in the metro (2025) and 74,710 in NYC, with an NYC entry wage of $121,283 and a median of $174,663. The time series shows the code change: `15-1132` (2018, SOC 2010 *applications* developers only) → `15-1256` (2019–20 hybrid, developers + QA testers) → `15-1252` (2021+). The 2018→2019 jump is a definition change, not growth.
- Cross-product consistency: NYS DOL's projections workbook (notebook 02) shows the same NYC 2026 mean wage ($95,518) as this table, so NYS DOL uses one wage vintage across products.

## 8. Fitness for purpose

| Question | Source | Fit | Why |
|---|---|---|---|
| Q4: what does a job pay at the start in NYC | NYS DOL | Use | The only source with an NYC-only, entry-level wage. Caveats: entry = mean of the bottom third (not a "new grad" wage); floored at the minimum wage; aged to Q1 2026 |
| **Q4: pay over time** | OEWS metro | Use with limits | 8 years of percentiles. But it's the metro, not NYC; codes change in 2019 and 2021; top-coding stops in 2025; the 2020–21 mean is inflated by composition |
| **Q1: jobs over time** | OEWS metro | partial | Metro, not NYC; excludes the self-employed; 3-year pooled sample, so it lags turning points |
| Wage *distribution* (how spread is pay) | OEWS metro | yes | 10th–90th percentile; NYS DOL only has mean/median/entry/experienced |
| Current NYC employment by job | NYS DOL | yes | 4.60M NYC jobs by SOC. Also the share weights for the decision D1 projection fill (decision log) |
| Anything below NYC (borough) | — | no | Neither source publishes boroughs |
| Real (inflation-adjusted) pay | — | Needs CPI (notebook 10) | Both are nominal |

## 9. Issues list and proposed staging tables

| # | Issue | Source · rows affected | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | Column names and set change by year (2018 layout; case; `OCC_GROUP`/`O_GROUP`, `LOC QUOTIENT`) | OEWS · all years | Normalize names (strip, upper, `_`); per-year rename map; assert the final column set |
| 2 | Excel lock files inside the 2022 zip | OEWS · 2022 | Select the member whose name starts with `MSA_` |
| 3 | Reference year only in the file name | OEWS · all | `year` from `oesmYYma`; cross-check against the manifest note |
| 4 | Symbols in numeric columns: `**` employment n/a, `*` wage n/a, `#` top-coded | OEWS · per year: 16–29 jobs `**`; 8–16 jobs `*` (annual), ~65 (hourly); 170–290 `#` cells (2018–24), 0 in 2025 | `**`/`*` → `NULL`; `#` → `NULL` + `is_top_coded = true` + `top_code_value` (100/hr ≤ 2021, 115/hr 2022–24) |
| 5 | Top-coding stops in 2025 without notice | OEWS · 2025 (68 jobs ≥ $239,200) | Keep the values; flag the series break in the year dimension (`top_coded = false` for 2025); footnote high-wage trends |
| 6 | Three SOC systems: 2010 (2018), hybrid (2019–20), 2018 (2021+) | OEWS · 2018–2020 | Keep the original code + `soc_edition`; map in core via SOC 2010→2018 + hybrid bridge (nb 06, shared with projections) |
| 7 | Metro redefined in 2024 (Pike County PA dropped), same code 35620 | OEWS · 2024–25 | Keep `area_title` per year; footnote the trend |
| 8 | Metro ≠ NYC (NYC ≈ 48% of jobs) | OEWS · all | `area_code = 35620`, `area_name = 'New York metro area'` in the area dimension; never label it NYC |
| 9 | Annual-only / hourly-only jobs (`ANNUAL`, `HOURLY` = TRUE) | OEWS · ~65 jobs | Keep as booleans `annual_only`, `hourly_only`; use annual wages for every comparison |
| 10 | Padded column names | NYS DOL · 8 of 10 columns | Strip names (cells are clean; strip anyway as a guard) |
| 11 | Blank employment (suppressed) / blank experienced wage | NYS DOL · 117 / 6 NYC rows | `NULL`, not zero |
| 12 | Wages floored at the 2026 minimum wage ($35,360 NYC) | NYS DOL · 22 NYC jobs at the floor | Flag `entry_wage_at_floor` (and `median_at_floor`) |
| 13 | Wages aged to Q1 2026; survey reference May 2025 | NYS DOL · all | Store `wage_reference = '2026Q1'` and `survey_reference = 'May 2025'`; don't mix with OEWS in one chart without saying so |
| 14 | One vintage, overwritten yearly on the portal | NYS DOL · all | `vintage` column from the manifest; keep each downloaded vintage if we re-download |
| 15 | COVID composition effect on 2020–21 wages | OEWS · 2020–21 | Document; growth KPIs use 2019 and 2022+ endpoints, or real wages (nb 10) |

**Proposed staging table 1:** `stg_bls__oews_metro_occupations`
- **Grain / key:** one row per `year × occ_code`, NY metro only, cross-industry. Key `(year, occ_code)`, asserted unique.
- **Columns:** `year` int, `area_code` text, `area_title` text, `occ_code` text, `occ_title` text, `soc_edition` text (`2010`/`hybrid`/`2018`), `occ_level` text (`total`/`major`/`detailed`), `employment` int, `employment_rse_pct` numeric, `jobs_per_1000` numeric, `location_quotient` numeric, `hourly_mean`, `annual_mean`, `mean_rse_pct`, `hourly_p10…p90`, `annual_p10…p90` numeric, `annual_only` bool, `hourly_only` bool, `is_top_coded` bool (any wage `#`), `source_file` text.
- **Filter:** `AREA = 35620` from each `MSA_` workbook.

**Proposed staging table 2:** `stg_nysdol__occupation_wages`
- **Grain / key:** one row per `area_code × soc_code`, NYC only for v1 (other regions kept out of scope). Key asserted unique.
- **Columns:** `vintage` text, `wage_reference` text (`2026Q1`), `area_code` text, `area_name` text, `soc_code` text, `soc_title` text, `soc_level` text, `employment` int, `mean_wage`, `median_wage`, `entry_wage`, `experienced_wage` int, `entry_wage_at_floor` bool, `median_at_floor` bool, `source_file` text.
- **Filter:** `Area Name = 'New York City'`.

In [19]:
# Row accounting for the proposed staging filters, recomputed rather than typed
print(f"OEWS NY metro rows (8 years): {len(oews):,} | per year: {oews.groupby('year').size().to_dict()}")
assert not oews.duplicated(["year", "OCC_CODE"]).any()
print(f"NYS DOL rows in: {len(wages):,} -> NYC rows out: {len(nyc_w):,}")
assert not nyc_w.duplicated("Standard Occupational Code").any()

OEWS NY metro rows (8 years): 6,281 | per year: {2018: 773, 2019: 759, 2020: 756, 2021: 804, 2022: 800, 2023: 803, 2024: 794, 2025: 792}
NYS DOL rows in: 7,036 -> NYC rows out: 702
